# Schema Mapping Deep Dive

Schema mapping is the process of aligning columns from your source data (EHR tables, CSV exports, legacy databases) to the standardized tables and columns in the OMOP Common Data Model (CDM) v5.4.

This notebook covers:

1. Setting up a project and ingesting source data
2. Running schema mapping and inspecting the results
3. Detailed inspection of `SchemaMappingItem` objects
4. Reviewing, approving, rejecting, and overriding individual mappings
5. Finalizing and exporting the schema map
6. An overview of OMOP CDM v5.4 target tables

Schema mapping is the second stage in the Portiere pipeline (after ingestion and profiling). It determines *where* your source data lands in the CDM -- concept mapping (covered in other notebooks) then determines *what* standard concepts the values map to.

> **Note:** Schema mapping does not use the `vocabularies` parameter. Vocabularies control which standard concept sets are searched during **concept mapping** (see [07_concept_mapping_deep_dive](./07_concept_mapping_deep_dive.ipynb)). Schema mapping instead matches source column names and data patterns against OMOP CDM table and column definitions.

## 1. Project Setup

First, initialize Portiere and create a project. We will use the hybrid knowledge backend for best accuracy and set schema mapping thresholds appropriate for a careful review workflow.

In [1]:
from pathlib import Path
from portiere.knowledge import build_knowledge_layer

ATHENA_DIR = Path("example_assets/vocabulary_download_v5")
VOCAB_DIR  = ATHENA_DIR / "indexes"
BM25S_CORPUS = VOCAB_DIR / "concepts.json"
FAISS_INDEX  = VOCAB_DIR / "concepts.index"
FAISS_META   = VOCAB_DIR / "concepts.meta.json"

# Build indexes from Athena on first run (~10-30 min for FAISS embedding)
if not BM25S_CORPUS.exists():
    paths = build_knowledge_layer(
        athena_path=ATHENA_DIR,
        output_path=VOCAB_DIR,
        backend="hybrid",
        vocabularies=["SNOMED", "LOINC", "RxNorm", "ICD10CM"],
    )
    print(f"Built indexes: {paths}")
else:
    print(f"Using existing indexes at {VOCAB_DIR}")

Using existing indexes at example_assets/vocabulary_download_v5/indexes


In [2]:
import portiere
from portiere.config import PortiereConfig, KnowledgeLayerConfig, ThresholdsConfig
from portiere.config import SchemaMappingThresholds, ConceptMappingThresholds, ValidationThresholds
from portiere.config import EmbeddingConfig, RerankerConfig
from portiere.engines import PolarsEngine

config = PortiereConfig(
    knowledge_layer=KnowledgeLayerConfig(
        backend="hybrid",
        bm25s_corpus_path=str(BM25S_CORPUS),
        faiss_index_path=str(FAISS_INDEX),
        faiss_metadata_path=str(FAISS_META),
        fusion_method="rrf",
        rrf_k=60,
    ),
    # Embedding and reranker default to HuggingFace + SapBERT/ms-marco.
    # Customize with EmbeddingConfig/RerankerConfig for other providers:
    # embedding=EmbeddingConfig(provider="ollama", model="nomic-embed-text"),
    # reranker=RerankerConfig(provider="none"),
    thresholds=ThresholdsConfig(
        schema_mapping=SchemaMappingThresholds(auto_accept=0.90, needs_review=0.70),
    ),
)

project = portiere.init(name="schema_mapping_demo", engine=PolarsEngine(), config=config)

print(f"Project: {project.name}")
print(f"Embedding: {config.embedding.provider}/{config.embedding.model}")
print(f"Reranker: {config.reranker.provider}/{config.reranker.model}")
print(f"Schema auto-accept threshold: {config.thresholds.schema_mapping.auto_accept}")
print(f"Schema needs-review threshold: {config.thresholds.schema_mapping.needs_review}")

2026-04-16 00:25:28 [info     ] PolarsEngine initialized      


2026-04-16 00:25:28 [debug    ] local_storage.initialized      base_dir=/Users/kuangsmacbook/.portiere/projects


Project: schema_mapping_demo
Embedding: huggingface/cambridgeltl/SapBERT-from-PubMedBERT-fulltext
Reranker: huggingface/cross-encoder/ms-marco-MiniLM-L-6-v2
Schema auto-accept threshold: 0.9
Schema needs-review threshold: 0.7


## 2. Add Source Data and Map Schema

Add a CSV file representing a patients table from a source EHR system. Portiere will profile the data and then attempt to map each column to an OMOP CDM target table and column.

In [3]:
# Add a source file -- Portiere automatically profiles columns, data types, and distributions
source = project.add_source("example_assets/06_schema_mapping_deep_dive/sample_patients.csv")

# Run schema mapping
schema_map = project.map_schema(source)

# High-level summary
summary = schema_map.summary()
print("Schema Mapping Summary:")
print(f"  Total columns: {summary['total']}")
print(f"  Auto-accepted: {summary['auto_accepted']}")
print(f"  Needs review: {summary['needs_review']}")
print(f"  Approved (manually): {summary['approved']}")
print(f"  Unmapped: {summary['unmapped']}")
print(f"  Auto-accept rate: {summary['auto_rate']:.1%}")

2026-04-16 00:25:28 [info     ] project.source_added           project=schema_mapping_demo source=sample_patients


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

2026-04-16 00:25:30 [info     ] gx_profiler.profiled           columns=11 rows=15 source=sample_patients


2026-04-16 00:25:30 [info     ] project.profiled               source=sample_patients


2026-04-16 00:25:30 [info     ] Stage 2: Schema mapping (local) embedding_model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext embedding_provider=huggingface target_model=omop_cdm_v5.4


...

2026-04-16 00:25:32 [warning  ] local_schema_mapper.init_failed: sentence-transformers is required for HuggingFace embeddings. Install with: pip install sentence-transformers


2026-04-16 00:25:32 [info     ] Stage 2 complete               auto=10 review=0 total=11


2026-04-16 00:25:32 [info     ] local_storage.schema_mapping_saved items_count=11 project=schema_mapping_demo


2026-04-16 00:25:32 [info     ] project.schema_mapped          auto=10 project=schema_mapping_demo total=11


Schema Mapping Summary:
  Total columns: 11
  Auto-accepted: 10
  Needs review: 0
  Approved (manually): 0
  Unmapped: 1
  Auto-accept rate: 9090.9%


## 3. Inspecting SchemaMappingItem Details

Each column in the source data produces a `SchemaMappingItem` with the following attributes:

| Attribute | Type | Description |
|-----------|------|-------------|
| `source_column` | str | The original column name from the source data |
| `source_table` | str | The original table name (defaults to empty string) |
| `target_table` | Optional[str] | The matched OMOP CDM table (e.g., `person`, `visit_occurrence`) |
| `target_column` | Optional[str] | The matched OMOP CDM column (e.g., `birth_datetime`, `visit_start_date`) |
| `confidence` | float | Confidence score between 0.0 and 1.0 |
| `status` | MappingStatus | One of: AUTO_ACCEPTED, NEEDS_REVIEW, APPROVED, REJECTED, OVERRIDDEN, UNMAPPED |
| `candidates` | list[dict] | Ranked list of candidate mappings with their scores |

Let us iterate through all items and display their details.

In [4]:
for item in schema_map.items:
    print(f"Column: {item.source_column}")
    print(f"  -> {item.target_table}.{item.target_column}")
    print(f"  Confidence: {item.confidence:.2f}")
    print(f"  Status: {item.status.value}")
    if item.candidates:
        print(f"  Candidates:")
        for c in item.candidates[:3]:
            print(f"    - {c.get('target_table')}.{c.get('target_column')} ({c.get('confidence', 0):.2f})")
    print()

Column: patient_id
  -> person.person_id
  Confidence: 0.95
  Status: auto_accepted

Column: first_name
  -> person.person_source_value
  Confidence: 0.95
  Status: auto_accepted

Column: last_name
  -> person.person_source_value
  Confidence: 0.50
  Status: unmapped

Column: date_of_birth
  -> person.birth_datetime
  Confidence: 0.95
  Status: auto_accepted

Column: gender
  -> person.gender_concept_id
  Confidence: 0.95
  Status: auto_accepted

Column: race
  -> person.race_concept_id
  Confidence: 0.95
  Status: auto_accepted

Column: ethnicity
  -> person.ethnicity_concept_id
  Confidence: 0.95
  Status: auto_accepted

Column: address
  -> location.address_1
  Confidence: 0.95
  Status: auto_accepted

Column: city
  -> location.city
  Confidence: 0.95
  Status: auto_accepted

Column: state
  -> location.state
  Confidence: 0.95
  Status: auto_accepted

Column: zip_code
  -> location.zip
  Confidence: 0.95
  Status: auto_accepted



## 4. Filtering by Routing Tier

You can retrieve items by their routing tier to focus your review on the items that need attention.

In [5]:
# Items that were auto-accepted (high confidence, no review needed)
auto_items = schema_map.auto_accepted()
print(f"Auto-accepted ({len(auto_items)} items):")
for item in auto_items:
    print(f"  {item.source_column} -> {item.target_table}.{item.target_column} "
          f"[{item.confidence:.2f}]")

print()

# Items that need human review (moderate confidence)
review_items = schema_map.needs_review()
print(f"Needs review ({len(review_items)} items):")
for item in review_items:
    print(f"  {item.source_column} -> {item.target_table}.{item.target_column} "
          f"[{item.confidence:.2f}]")

Auto-accepted (10 items):
  patient_id -> person.person_id [0.95]
  first_name -> person.person_source_value [0.95]
  date_of_birth -> person.birth_datetime [0.95]
  gender -> person.gender_concept_id [0.95]
  race -> person.race_concept_id [0.95]
  ethnicity -> person.ethnicity_concept_id [0.95]
  address -> location.address_1 [0.95]
  city -> location.city [0.95]
  state -> location.state [0.95]
  zip_code -> location.zip [0.95]

Needs review (0 items):


## 5. Approve, Reject, and Override Individual Mappings

For items in the "needs review" tier, you have three options:

- **approve()** -- Accept the current mapping as correct.
- **approve(target_table=..., target_column=...)** -- Accept but override with a different target.
- **reject()** -- Mark the mapping as rejected (item becomes UNMAPPED).

These actions change the item's `status` field accordingly.

In [6]:
review_items = schema_map.needs_review()

if len(review_items) >= 1:
    # Example 1: Approve the mapping as-is
    item = review_items[0]
    print(f"Before: {item.source_column} -> {item.target_table}.{item.target_column} "
          f"(status: {item.status.value})")
    item.approve()
    print(f"After:  {item.source_column} -> {item.target_table}.{item.target_column} "
          f"(status: {item.status.value})")
    print()

if len(review_items) >= 2:
    # Example 2: Override with a different target
    item = review_items[1]
    print(f"Before: {item.source_column} -> {item.target_table}.{item.target_column} "
          f"(status: {item.status.value})")
    item.approve(target_table="person", target_column="birth_datetime")
    print(f"After:  {item.source_column} -> {item.target_table}.{item.target_column} "
          f"(status: {item.status.value})")
    print()

if len(review_items) >= 3:
    # Example 3: Reject a mapping
    item = review_items[2]
    print(f"Before: {item.source_column} -> {item.target_table}.{item.target_column} "
          f"(status: {item.status.value})")
    item.reject()
    print(f"After:  {item.source_column} -> target: None "
          f"(status: {item.status.value})")

## 6. Batch Approval and Finalization

After reviewing individual items, you can approve all remaining review items in bulk, then finalize the schema mapping. Finalization locks the mapping and prepares it for the concept mapping stage.

In [7]:
# Approve all remaining items that still need review
schema_map.approve_all()

# Verify the updated summary
summary = schema_map.summary()
print("Updated Schema Mapping Summary:")
print(f"  Total columns: {summary['total']}")
print(f"  Auto-accepted: {summary['auto_accepted']}")
print(f"  Needs review: {summary['needs_review']}")
print(f"  Approved: {summary['approved']}")
print(f"  Unmapped: {summary['unmapped']}")
print(f"  Auto-accept rate: {summary['auto_rate']:.1%}")
print()

# Finalize the schema mapping
schema_map.finalize()
print("Schema mapping has been finalized.")
print("The mapping is now locked and ready for the concept mapping stage.")

Updated Schema Mapping Summary:
  Total columns: 11
  Auto-accepted: 10
  Needs review: 0
  Approved: 0
  Unmapped: 1
  Auto-accept rate: 9090.9%

Schema mapping has been finalized.
The mapping is now locked and ready for the concept mapping stage.


## 7. Exporting the Schema Map

You can export the finalized schema mapping for documentation or downstream tooling.

In [8]:
# Display the final mapping as a table
print(f"{'Source Column':<25} {'Target Table':<25} {'Target Column':<25} {'Confidence':<12} {'Status'}")
print("-" * 100)

for item in schema_map.items:
    target_table = item.target_table or "(unmapped)"
    target_column = item.target_column or "(unmapped)"
    print(f"{item.source_column:<25} {target_table:<25} {target_column:<25} "
          f"{item.confidence:<12.2f} {item.status.value}")

Source Column             Target Table              Target Column             Confidence   Status
----------------------------------------------------------------------------------------------------
patient_id                person                    person_id                 0.95         auto_accepted
first_name                person                    person_source_value       0.95         auto_accepted
last_name                 person                    person_source_value       0.50         unmapped
date_of_birth             person                    birth_datetime            0.95         auto_accepted
gender                    person                    gender_concept_id         0.95         auto_accepted
race                      person                    race_concept_id           0.95         auto_accepted
ethnicity                 person                    ethnicity_concept_id      0.95         auto_accepted
address                   location                  address_1          

## OMOP CDM v5.4 Target Tables Overview

When Portiere maps your source columns, it targets the OMOP Common Data Model v5.4 tables. Below is a summary of the most commonly used clinical data tables.

### Clinical Data Tables

| Table | Description | Key Columns |
|-------|-------------|-------------|
| **person** | Demographics for each patient | person_id, gender_concept_id, year_of_birth, month_of_birth, day_of_birth, birth_datetime, race_concept_id, ethnicity_concept_id |
| **visit_occurrence** | Encounters and visits | visit_occurrence_id, person_id, visit_concept_id, visit_start_date, visit_start_datetime, visit_end_date, visit_type_concept_id |
| **condition_occurrence** | Diagnoses and conditions | condition_occurrence_id, person_id, condition_concept_id, condition_start_date, condition_type_concept_id, condition_source_value |
| **drug_exposure** | Medication prescriptions and administrations | drug_exposure_id, person_id, drug_concept_id, drug_exposure_start_date, drug_type_concept_id, quantity, days_supply |
| **procedure_occurrence** | Clinical procedures | procedure_occurrence_id, person_id, procedure_concept_id, procedure_date, procedure_type_concept_id |
| **measurement** | Lab results, vital signs, and other measurements | measurement_id, person_id, measurement_concept_id, measurement_date, value_as_number, unit_concept_id |
| **observation** | Clinical observations not captured elsewhere | observation_id, person_id, observation_concept_id, observation_date, value_as_string, value_as_number |
| **death** | Patient death information | person_id, death_date, death_datetime, death_type_concept_id, cause_concept_id |

### Health System Data Tables

| Table | Description |
|-------|-------------|
| **location** | Geographic location information (addresses) |
| **care_site** | Facilities and departments where care is delivered |
| **provider** | Healthcare providers (physicians, nurses, etc.) |

### Vocabulary Tables

| Table | Description |
|-------|-------------|
| **concept** | Master vocabulary table with all standard concepts |
| **concept_relationship** | Relationships between concepts (maps_to, is_a, etc.) |
| **source_to_concept_map** | Custom mappings from source codes to standard concepts |

### Key Points for Schema Mapping

- Every clinical event table links back to `person` via `person_id`.
- Most tables have a `*_concept_id` column that holds the standard concept identifier and a corresponding `*_source_value` column that holds the original source code.
- Date columns come in pairs: `*_date` (DATE type) and `*_datetime` (DATETIME type). Portiere maps to the appropriate one based on your source data precision.
- `*_type_concept_id` columns indicate the provenance of the data (e.g., EHR, claim, self-reported). These are typically set during ETL generation, not schema mapping.

For the complete CDM v5.4 specification, refer to the [OHDSI CDM documentation](https://ohdsi.github.io/CommonDataModel/).